In [ ]:
%matplotlib inline

# Board Game Simulation

## Volcano Rush

### Abstract

This study uses Monte Carlo simulation with rule-based agents to evaluate the balance of *Volcano Rush*, a semi-cooperative board game for 6-8 players. Thousands of games (1,000-10,000 per scenario) are simulated across player counts of 6, 7, and 8, recording win rates, individual scores by character role, and resource consumption patterns.

Win rates cluster in a narrow band: 45.7% (6 players), 49.8% (7 players), 52.1% (8 players). Seven and eight players sit inside the 50-65% target; six players is slightly below, edging toward a hard configuration. Volcano tension is tight at every count with a median of 3 cards remaining on wins. Character balance reveals a sharp gap: Builder achieves a 29.2% personal win rate, close to double the 16.7% uniform baseline, while Craftsman, Cook, and Fire Starter cluster near baseline (16.0-16.3%) and Sailor (10.6%) + Gatherer (11.7%) trail. Builder also dominates the group-win-effect metric (32.8%), followed by Sailor (22.2%) whose support contribution through lesser-evil complication draws is invisible in the personal-score view. Resource analysis shows Wood and Rope causing roughly equal mission-failure pressure (1.01 / 1.06 per game), Stone running slack (0.49 / game), and the any-resource rule contributing little (0.20 / game).

These results suggest two remaining design adjustments: reducing Builder's scoring advantage (e.g. narrowing the wood discount or raising wood demand in wood-heavy missions), and surfacing support-role contributions (Sailor, Craftsman) in the personal score so they are not only visible in group win effect.


### Introduction

Game balance is considered critical to the success of any game, yet there is no universal consensus on what it actually means [1]. It can be broadly interpreted as the process of tuning a game's rules, difficulty, play time, resources, and role abilities so that no single strategy, character, or configuration dominates the experience. For multiplayer games in particular, balance has several dimensions: ensuring no starting position is inherently advantageous, preventing any strategy from being strictly dominant, and calibrating the cost-to-benefit ratio across different game objects [1].

As a semi-cooperative board game, *Volcano Rush* faces a specific balancing challenge: the group must share a win condition (escaping the volcano), while players also compete for individual scores. A well-balanced game therefore needs a group win rate that feels achievable but not trivial, and individual scores that are not dominated by a single character role. The game supports 6 to 8 players, and balance must hold across all three configurations.

As with most game design problems, there is no deterministic recipe for achieving balance. *Volcano Rush* is both strategic and social, and raw numbers can sometimes mislead. Simulation is not a substitute for playtesting, but it provides a principled starting point for identifying where imbalances may arise before bringing the game to the table.


### Related Work

#### Monte Carlo simulation

Monte Carlo simulation estimates probabilities by running many random trials instead of solving mathematical formulas, which can be extremely difficult for complex games. Running $n$ simulated games under a given strategy $M$ yields an empirical win probability:

We define an indicator variable for each simulation:

* $X_i = 1$ if simulation $i$ results in a win  
* $X_i = 0$ otherwise  

The empirical win probability after $n$ simulations is:

$$
\hat{p}_n(M) = \frac{1}{n} \sum_{i=1}^{n} X_i
$$

As the number of simulations increases, this estimate converges to the true win probability:

$$
p(M) = \lim_{n \to \infty} \frac{1}{n} \sum_{i=1}^{n} X_i
$$

This follows from the **Law of Large Numbers**, which states that the sample average converges to the expected value (true probability) as the number of trials grows. This makes Monte Carlo methods a good fit for games with many possible situations, where computing the exact answer is not practical.

#### Monte Carlo Tree Search (MCTS)

Monte Carlo methods are also widely used in game AI through Monte Carlo Tree Search (MCTS) [2], which builds a tree of possible future game states and tests them with simulations. 
At each step, the algorithm faces a trade-off between two ideas:
* *exploitation* - choosing moves that have already shown good results  
* *exploration* - trying moves that have not been tested much but might be better  

This balance is handled by the **Upper Confidence Bound for Trees** (UCT) formula:

$$ \mathrm{UCT} = \frac{W_i}{N_i} + c \sqrt{\frac{\ln N_p}{N_i}} $$

The formula combines two parts:
* $\frac{W_i}{N_i}$ - the **exploitation term**, which represents how successful a move has been so far (its average win rate)
* $c \sqrt{\frac{\ln N_p}{N_i}}$ - the **exploration term**, which gives a higher value to moves that have been tried fewer times, encouraging the algorithm to explore them
* $c$ - a constant that controls the balance between exploration and exploitation (higher values lead to more exploration)

In practice, MCTS does not explore all possible game paths. Instead, it gradually builds the search tree through repeated simulations. At the beginning, it gathers initial data by trying different possible moves. As more simulations are performed, the algorithm starts to focus more on the moves that show better results, exploring them in greater depth. However, less-explored moves are still occasionally tested due to the exploration term in the UCT formula, ensuring that potentially good alternatives are not missed.

In this work, however, we use simpler repeated simulations with rule-based agents to study game balance rather than to improve decision-making [2].

#### Agent-Based Modeling (ABM)

In Agent-Based Modeling (ABM), each agent follows a set of simple rules and interacts with other agents and the world around it [3]. Over time, these small interactions can produce complex group behaviors that would be hard to predict just by looking at the rules alone.

Compared to pure Monte Carlo simulation, ABM allows for more realistic behavior. Monte Carlo is fully random, but in ABM, agents can follow rules that reflect real human decisions. This approach has been used in research - for example, interviewing participants after a game experiment and using their answers to design the decision rules for ABM agents [5]. For *Volcano Rush*, we could do the same - interview players after a playtest session and use what we learn to make the agents behave more like real people [4].

Traditional game theory assumes that players always make the best possible decision to get the best outcome for themselves. But in real life, people are often influenced by emotions, habits, and not having all the information [3]. This means game theory models do not always reflect how real people actually play.

### Data

This study uses no external dataset. All data is generated by the simulation itself - each run of the simulation produces one game record. The input to the simulation is a fixed set of game parameters taken directly from the game rules.

#### Game Parameters

The resource deck contains **60 cards** split equally between three types: Wood, Stone, and Rope (20 each). The deck is reshuffled and reused when it runs out. The camp has two **shared tools** - Knife and Vessel - which can become damaged and must be repaired before use.

There are **6 character roles**: Builder, Fire Starter, Craftsman, Cook, Gatherer, and Sailor. Each has a unique ability that affects resource requirements, point rewards, or complication handling. For 6 players, all six roles are used once. For 7-8 players, one or two non-Craftsman roles are drawn at random to fill the extra seats.

The **mission pool** consists of 8 non-boat missions and up to 5 boat missions. At any time, exactly 3 missions are active and new missions are drawn when one is completed or discarded. The number of boat missions in play - and the number of boat parts needed to win - depends on player count:

| Players | Boat Parts Required | Active Boat Missions |
|---|---|---|
| 6 | 3 | Cut the Keel, Assemble the Hull, Raise the Mast |
| 7 | 4 | + Make the Sail |
| 8 | 5 | + Fit the Rudder |

Boat parts must be completed in a fixed construction order: Cut the Keel -> Assemble the Hull -> Raise the Mast -> Make the Sail -> Fit the Rudder.

The **Complication deck** contains 11 cards (including 2 neutral cards with no effect) and is reshuffled when exhausted. Complication effects are charged to **each participant individually** (no pooled payment). The **Volcano deck** contains 11 penalty cards with the Eruption card fixed at the bottom - when it is drawn, the game ends in a loss. Volcano cards that carry extra resource costs (Rain and Mud, Lava Flow) are also charged per participant.
Card draws from the resource, complication, and volcano decks are modeled as stochastic processes with uniform random sampling.

#### Simulation Assumptions

Where the game rules leave room for interpretation, the following assumptions are applied:

| Rule Area | Assumption |
|---|---|
| Active player | Rotates through the player list each round; the active player decides the round's action (mission, shuffle, or fallback volcano draw) |
| Mission selection | The active player selects the mission unilaterally using their character's voting preference |
| Participant selection | Every non-exhausted candidate is scored by affordability (EXACT / SURPLUS), character-ability relevance, a Craftsman-repair penalty, and a soft competition penalty vs. the active player; the top-N by score are selected |
| Resource payment | Each participant pays their own combined per-player cost (base mission + character discount + complication extras + volcano extras) from their own hand; there is no pooled group contribution |
| Participation cost | A participant must fully cover their own combined per-player cost or the mission fails |
| Shuffle decision | If Panic is pending and all active missions are boat parts, or if all active missions are boat parts but the next-needed part is missing, the active player shuffles (provided they can pay the cost) |
| Shuffle cost | Costs the active player 1 resource card of any type; cannot shuffle with 0 resources |
| Exhaustion | Applies to all participants after resolution, regardless of success or failure (unless bonus_on_success.no_exhaustion was triggered) |
| Craftsman repair | Automatic when not participating: if not exhausted, has Stone, and a tool is damaged with no repair in progress. Starts round N, tool unavailable round N+1, available again round N+2 |
| Sailor - lesser evil | The less severe Complication card is chosen using a severity scoring function |
| Participant count | Treated as an exact requirement, no additional participants allowed |
| Damaged tool | A mission requiring a damaged tool automatically fails |
| Night Anxiety | Requires 1 non-participant helper; if none available, mission fails |
| Gather - default | Any non-participant automatically gathers (after Craftsman repair check) |
| Gather - non-Gatherer | Drawing 1 resource via Gather does not cause Exhaustion |
| Gather - Gatherer role | Using the boosted draw (3 resources when hand size < 3) causes Exhaustion |

#### Output Variables

Each simulation run records: game outcome (win/loss), number of rounds played, individual score per player, win rate per role (role dominance), number of boat parts completed, and number of volcano cards remaining when the game ends.


### Methods

The simulation runs each game as a sequence of rounds following the round structure described in `docs/game_rules.md`. Thousands of games are run per configuration, and outcomes are recorded to estimate win rates and individual scores empirically.

#### Agent Strategy

Each round is led by a rotating **active player** who makes key decisions. The remaining players follow simple rules based on whether they are selected as participants.

**Active player decision (start of each round)**
The active player chooses one of the following actions:
1. **Choose a mission** - select one of the 3 active missions using their character preference (e.g. Builder prefers Wood-requiring missions, Cook prefers Food missions, Sailor prefers Boat missions).
2. **Shuffle the mission deck** - reshuffle all undrawn missions. Costs 1 resource card from the active player. No mission is attempted and no gathering occurs this round.

The shuffle decision applies in priority order:
* Panic volcano card is pending AND all active missions are boat parts (boats are banned under Panic) - shuffle.
* All active missions are boat parts, but the next-needed boat part is missing from them - shuffle.
* Otherwise - choose a mission.

Both shuffle triggers require the active player to have a resource to pay the shuffle cost; if not, the round falls back to choosing a mission.

**Participant selection (when a mission is chosen)**
Every non-exhausted candidate is scored by a heuristic that combines:
* Affordability of the per-player cost - EXACT hand scores +3, SURPLUS hand scores +5, unaffordable hands are excluded.
* Character-ability relevance to the mission (+10) - Cook on food, Fire Starter on fire, Sailor on boat, Builder on wood-requiring.
* Craftsman-repair penalty (-20) when the Craftsman could repair a damaged tool instead of participating.
* Soft competition penalty (-1 per score lead point, floored at -8) applied ~75% of calls.

The top `players_count` candidates are selected; the active player is prepended when they can afford their own cost.

**Non-participant actions**
* **Craftsman** (if not Exhausted, has Stone, and a tool is damaged with no repair in progress): automatically repairs a tool (+1 point), then gathers.
* All other non-participants: **Gather** - draw 1 random resource card.
  * **Gatherer role**: draws 3 resources when not Exhausted and holds fewer than 3 cards; becomes Exhausted afterward.

**Boat mission priority**
* When the volcano deck has 4 or fewer cards remaining, the active player gives maximum priority to boat missions in their mission selection.

#### Simulation Scenarios

Games are run across all supported player counts (6-8). Character assignment follows the game rules: all 6 roles are used exactly once in 6-player games; non-Craftsman roles are repeated at random to fill the extra seats in 7-8 player games. Each scenario runs N = 1,000-10,000 games. Sensitivity analysis varies scoring weights and urgent-volcano threshold.

#### Statistical Analysis

Win rates per scenario are estimated empirically and reported with 95% Wald confidence intervals.


#### Simulation Engine

The engine exposes two public functions:

* `run_game(player_count, ...)` - runs a single game and returns a `GameRecord`
* `run_scenario(player_count, n_games, base_seed, ...)` - runs `n_games` games and returns a list of `GameRecord`. When `base_seed` is provided, each game is seeded as `base_seed + i`, making the entire scenario reproducible.

`GameRecord` captures the outcome, number of rounds played, character list, final scores per character, boat parts built vs. required, and volcano cards remaining at game end.

Each game runs as a sequence of rounds via `run_round`, which executes these steps in order:

**active player decision → mission selection → participant selection → non-participant actions → complication draw → mission resolution → exhaustion → gather → end-of-round housekeeping**

End-of-round housekeeping (in `GameState.end_round`) covers: consuming a pending Panic card, replacing the completed mission with a fresh pool draw, checking the win condition, and rotating the active player.

If the active player chooses to shuffle the mission deck, the round ends immediately after the shuffle cost is paid and the deck is reshuffled - all intermediate steps are skipped.

#### Agent Heuristics

Each round is controlled by three agent functions that encode hand-coded decision rules.

**Active player decision - `decide_mission_action`**

Returns `SHUFFLE_MISSIONS` in one of two cases:

1. Panic volcano card is pending AND all active missions are boat parts - the boat ban forces a shuffle.
2. All active missions are boat parts AND the next-needed boat part (by construction order) is missing - shuffling is preferred over building out of order.

Both cases require the active player to have a resource to pay the shuffle cost; otherwise the default `CHOOSE_MISSION` branch runs.

**Mission selection - `vote_for_mission`**

The active player selects a mission using their character preference:

* Panic pending: boat missions are filtered out.
* Urgent volcano (volcano deck ≤ `urgent_volcano_threshold`, default 4): the active player ignores character preference and selects the next-needed boat part if it is active; otherwise a random boat mission.
* Otherwise, character preference applies: Builder picks missions requiring ≥1 Wood, Fire Starter picks fire-type missions, Cook picks food-type missions, Sailor picks boat missions. Craftsman and Gatherer fall back to a random feasible pick.

**Participant selection - `active_player_select_participants`**

Every non-exhausted candidate is scored by `_participant_score`; the top `players_count` are taken, with ties broken by player index for stable selection. Scoring combines:

* `_AFFORD_SCORE[EXACT]` = 3 / `_AFFORD_SCORE[SURPLUS]` = 5 - SURPLUS is preferred because those players can absorb per-participant complication and volcano extras from their hand.
* `_ACTIVE_ABILITY_BONUS` (+10) when `CharacterStrategy.has_active_ability_on(mission)` returns True (Cook/food, Fire Starter/fire, Sailor/boat, Builder/wood-requiring).
* `_CRAFTSMAN_REPAIR_PENALTY` (-20) when the candidate is Craftsman and a tool needs repair.
* Competition penalty (-1 per point of lead over the active player's score, floored at -8), applied probabilistically (75% of calls) so the active player does not always optimize personal competition.

The active player is prepended to the shortlist when they can afford their own cost.

**Gather amount - `GathererStrategy.choose_non_participant_action`**

The Gatherer draws 3 resources when not exhausted and holds fewer than 3 cards (the draw then causes Exhaustion). All other non-participants draw 1 via the default `GatherAction`.


#### Simulations

* See [1_simulation_verbose_run.ipynb](simulations/1_simulation_verbose_run.ipynb).
* See [2_simulation_player_count_balance.ipynb](simulations/2_simulation_player_count_balance.ipynb).
* See [3_simulation_character_balance.ipynb](simulations/3_simulation_character_balance.ipynb).
* See [4_simulation_resource_efficiency.ipynb](simulations/4_simulation_resource_efficiency.ipynb).

### Discussion

The simulations reveal a more balanced picture than earlier iterations, but some structural issues remain:

* **Win rates are close to target.** 6 players (45.7%), 7 players (49.8%), 8 players (52.1%) - 7 and 8 are inside the 50-65% target, 6 sits just below. Previously 8-player was far above target at 75%; the per-participant cost model and the FIT_THE_RUDDER staffing bump corrected that.
* **Character balance is dominated by Builder.** Builder (29.2%) is nearly double the next tier (Craftsman 16.3%, Cook 16.2%, Fire Starter 16.0%). Sailor (10.6%) and Gatherer (11.7%) still trail. Builder's wood discount now applies across far more cost (per-participant complication and volcano extras), which inflates their personal score. This is a new imbalance introduced by the per-participant model.
* **Group win effect tells a different story.** Builder (32.8%) and Sailor (22.2%) are the top group contributors; Sailor's lesser-evil complication draws on boat missions are valuable even though they produce almost no personal points. The scoring system misses this.
* **Volcano tension is tight and stable.** Median 3 cards remaining on wins at every player count (was 3 -> 4 -> 6 previously). Eruption risk registers across all configurations.
* **Resource pressure is more even.** Wood (1.01 failures/game) and Rope (1.06) are now basically equal; Stone (0.49) runs slack but not dramatically so. The any-resource rule contributes only 0.20 failures/game.

### Limitations

* **Fixed strategies.** Agents use the same heuristics every game, so the simulation cannot measure strategy diversity - how many different paths to victory exist. A well-balanced game should have no strictly dominant strategy [1].
* **No communication model.** Real players negotiate who pays what, which missions to prioritize, and when to shuffle. Traditional game theory assumes optimal decision-making, but real players are influenced by emotions, habits, and incomplete information [3]. This social layer can shift balance in ways the simulation can't capture. Interviewing players after playtests and using their answers to design agent decision rules [4][5] would produce more realistic behavior.
* **Incomplete scoring visibility.** Characters with support roles (Sailor, Craftsman) are undervalued because their contributions are invisible to the current mission-point-only metrics. A hidden points system that tracks group and personal effectiveness would give a more complete picture.

### Further Work

* **Builder rebalance.** Builder's wood discount dominates personal scoring after the per-participant rework. Options: narrow the discount to the base mission cost only (not complication/volcano extras), halve the discount amount, or raise wood requirements on wood-heavy missions.
* **Support-role scoring.** Surface Sailor's lesser-evil contributions in the personal score, not only in group win effect. Candidate: +1 personal point per lesser-evil draw on a successful boat mission.
* **Six-player push.** 45.7% is slightly below the 50-65% target; reducing one complication's severity, or adjusting boat mission staffing, could bring six-player games into band.
* **First-player advantage.** The starting active player makes the first mission decision and scores first. Over many rounds the rotation evens out, but early-round advantage in resource spending and point accumulation could affect personal win rate. A balanced game should ensure no starting position is inherently advantageous [1]. A simulation comparing average score by player position (1st through 6th) would reveal whether starting order matters.
* **Strategy diversity (hardest).** Test whether different approaches lead to different outcomes. This requires either multiple distinct agent types or parameterized behavior weights (e.g., aggressive vs. conservative mission selection). Also test the urgent volcano flag at different thresholds.
* **Volcano card effects.** Track which card was in play when losses happen and which characters help survive specific effects. Some cards compound over time (e.g., Ash in the Air stacking exhaustion) and won't show as the last card before a loss but are still impactful - this needs cumulative card tracking per game.
* **Complication card effects.** Same loss-correlation approach. Most interesting: test modified decks with harsher effects repeated (e.g., double the Storm cards) and compare win rates. The Sailor's ability (draw 2 complications, keep the less severe one) is directly testable here.
* **Rule variant testing.** Test with adjusted resource requirements and adjusted boat-part staffing to tune balance further.
* **MCTS agents.** Replace rule-based heuristics with Monte Carlo Tree Search [2] for more realistic decision-making. Adds complexity but would produce more reliable balance estimates.
* **Engine tracking extension.** The volcano and complication card simulations require the engine to record which cards appeared and when. GameRecord needs to be extended with card event tracking before those simulations can run.


### References / Bibliography

[1] [Schell, J. (2009). Level 16: Game Balance. Game Design Concepts.](https://gamedesignconcepts.wordpress.com/2009/08/20/level-16-game-balance/)

[2] [Chaslot, G., Bakkes, S., Szita, I., & Spronck, P. (2008). Monte-Carlo Tree Search: A New Framework for Game AI. AIIDE.](https://sander.landofsand.com/publications/AIIDE08_Chaslot.pdf)

[3] [StudyGuides.com. (2026). Simulations in Game Theory.](https://studyguides.com/study-methods/overview/clzggcm6j0j8t8cfe9zugo2t8)

[4] [Dubois, E., Barreteau, O., & Souchère, V. (2013). An Agent-Based Model to Explore Game Setting Effects on Attitude Change During a Role Playing Game Session. Journal of Artificial Societies and Social Simulation, 16(1), 2.](https://www.jasss.org/16/1/2.html)

[5] [Yiannakoulias, N., Grignon, M., & Marshall, T. (2024). Parameterizing agent-based models using an online game. Computers, Environment and Urban Systems, 112, 102142.](https://www.sciencedirect.com/science/article/pii/S0198971524000711)

### Appendix

* **Appendix A:** General game rules (overview, resources, mechanics, scoring) - `docs/game_rules.md`
* **Appendix B:** Mission table - `docs/missions.md`
* **Appendix C:** Complication card descriptions - `docs/complication_cards.md`
* **Appendix D:** Volcano card descriptions - `docs/volcano_cards.md`
* **Appendix E:** Character ability reference - `docs/characters.md`